# CosyVoice 3 (Fun-CosyVoice3-0.5B) - Technical Report

## Overview

**CosyVoice 3** (Fun-CosyVoice3-0.5B-2512) is an advanced **LLM-based text-to-speech (TTS)** system developed by Alibaba's FunAudioLLM team. It is the third generation of the CosyVoice family, surpassing CosyVoice 2 in content consistency, speaker similarity, and prosody naturalness.

### Key Features
- **9 languages** (Chinese, English, Japanese, Korean, German, Spanish, French, Italian, Russian) and **18+ Chinese dialects/accents**
- **Zero-shot voice cloning** across languages
- **Pronunciation inpainting** with Chinese Pinyin and English CMU phonemes
- **Built-in text normalization** (no separate frontend needed)
- **Bi-streaming** support (text-in streaming + audio-out streaming, 150ms latency)
- **Instruct support** for controlling language, dialect, emotion, speed, volume

### Performance (CV3-Eval Benchmark)
| Model | Size | test-zh CER% | test-zh SS% | test-en WER% | test-en SS% |
|---|---|---|---|---|---|
| Human | - | 1.26 | 75.5 | 2.14 | 73.4 |
| CosyVoice2 | 0.5B | 1.45 | 75.7 | 2.57 | 65.9 |
| **Fun-CosyVoice3** | **0.5B** | **1.21** | **78.0** | **2.24** | **71.8** |
| Fun-CosyVoice3 RL | 0.5B | **0.81** | 77.4 | **1.68** | 69.5 |

CER = Character Error Rate, WER = Word Error Rate, SS = Speaker Similarity

---

# 1. High-Level Architecture

CosyVoice 3 follows a **three-stage pipeline**:

```
Text Input ──> [LLM (Qwen2-based)] ──> Speech Tokens ──> [Flow Matching (DiT)] ──> Mel Spectrogram ──> [HiFi-GAN Vocoder] ──> Waveform
```

The three core components are:

| Stage | Module | Class | Role |
|---|---|---|---|
| **1. LLM** | `cosyvoice.llm.llm.CosyVoice3LM` | Autoregressive LM | Converts text tokens into discrete speech tokens |
| **2. Flow** | `cosyvoice.flow.flow.CausalMaskedDiffWithDiT` | Conditional Flow Matching | Converts speech tokens into mel spectrograms |
| **3. HiFi-GAN** | `cosyvoice.hifigan.generator.CausalHiFTGenerator` | Neural Vocoder | Converts mel spectrograms into audio waveforms |

Additionally, the **Frontend** (`cosyvoice.cli.frontend.CosyVoiceFrontEnd`) handles text normalization, tokenization, speaker embedding extraction (CAM++), and speech token extraction.

In [ ]:
# The top-level class hierarchy: CosyVoice3 extends CosyVoice2 which extends CosyVoice
# File: cosyvoice/cli/cosyvoice.py

class CosyVoice:
    """Base class (CosyVoice v1) - uses TransformerLM + MaskedDiffWithXvec + HiFT"""
    pass

class CosyVoice2(CosyVoice):
    """CosyVoice v2 - uses Qwen2LM + CausalMaskedDiffWithXvec + CausalHiFT"""
    pass

class CosyVoice3(CosyVoice2):
    """CosyVoice v3 - uses CosyVoice3LM + CausalMaskedDiffWithDiT + CausalHiFT"""
    pass

# AutoModel factory dispatches based on which yaml file exists in model_dir:
def AutoModel(**kwargs):
    if os.path.exists(f"{kwargs['model_dir']}/cosyvoice3.yaml"):
        return CosyVoice3(**kwargs)
    elif os.path.exists(f"{kwargs['model_dir']}/cosyvoice2.yaml"):
        return CosyVoice2(**kwargs)
    elif os.path.exists(f"{kwargs['model_dir']}/cosyvoice.yaml"):
        return CosyVoice(**kwargs)

---

# 2. Stage 1: The LLM (Autoregressive Speech Token Predictor)

## 2.1 Architecture

The LLM component (`CosyVoice3LM`) is the core of the system. It takes text tokens as input and autoregressively predicts discrete **speech tokens** (from a vocabulary of 6561 FSQ tokens).

Key architectural choices in CosyVoice 3:

- **Backbone**: `Qwen2ForCausalLM` (a 0.5B parameter pretrained language model from Alibaba)
- **Text tokenizer**: Qwen2 tokenizer with extensive special tokens for phonemes, breath markers, laughter, etc.
- **Speech token vocabulary**: 6561 tokens (from a Finite Scalar Quantization / FSQ speech tokenizer) + special tokens (SOS, EOS, TASK_ID, FILL)
- **No separate text encoder** (unlike CosyVoice v1): text tokens are directly embedded via Qwen2's `embed_tokens` layer
- **Unified embedding space**: both text and speech tokens share the Qwen2 model's hidden dimension (896)

### Key Difference from CosyVoice 2
In CosyVoice 3, the `sos_emb` and `task_id_emb` are drawn from the **speech embedding** table rather than a separate `llm_embedding` table. The `llm_decoder` projects to `speech_token_size + 200` classes (with extra slots for future special tokens), and the speech embedding table is correspondingly sized.

In [ ]:
# File: cosyvoice/llm/llm.py — CosyVoice3LM initialization
# CosyVoice3LM inherits from Qwen2LM which inherits from TransformerLM

class CosyVoice3LM(Qwen2LM):
    def __init__(self, llm_input_size=896, llm_output_size=896,
                 speech_token_size=6561, llm=None, sampling=None,
                 length_normalized_loss=True, lsm_weight=0, mix_ratio=[5, 15]):
        
        # Special token IDs — note these are ABOVE the speech token range
        self.sos       = speech_token_size + 0   # 6561 — start-of-sequence
        self.eos_token = speech_token_size + 1   # 6562 — end-of-sequence
        self.task_id   = speech_token_size + 2   # 6563 — task separator
        self.fill_token = speech_token_size + 3  # 6564 — fill token (for bistream)

        # The LLM backbone is Qwen2ForCausalLM (0.5B parameters)
        self.llm = llm  # Qwen2Encoder wrapping Qwen2ForCausalLM

        # Decoder head: projects Qwen2 hidden states -> speech token logits
        # 6561 + 200 output classes (extra slots for special tokens)
        self.llm_decoder = nn.Linear(llm_output_size, speech_token_size + 200, bias=False)

        # Speech embedding: maps speech token IDs -> dense vectors for LLM input
        self.speech_embedding = nn.Embedding(speech_token_size + 200, llm_input_size)

        # Bi-stream interleaving ratio: 5 text tokens per 15 speech tokens
        self.mix_ratio = mix_ratio  # [5, 15]

        # Sampling strategy: Repetition-Aware Sampling (RAS)
        self.sampling = sampling  # ras_sampling(top_p=0.8, top_k=25, win_size=10, tau_r=0.1)

## 2.2 LLM Input Sequence Format

CosyVoice 3 supports two sequence formats for training, chosen randomly (50/50) per sample:

### Unistream (Traditional)
```
[SOS] [instruct_tokens] [text_tokens] [TASK_ID] [speech_tokens] [EOS]
```

### Bistream (Interleaved, new in v2/v3)
Text and speech tokens are interleaved in chunks of `mix_ratio = [5, 15]`:
```
[SOS] [instruct_tokens] [5 text] [15 speech] [FILL] [5 text] [15 speech] [FILL] ... [remaining text] [TASK_ID] [remaining speech] [EOS]
```

The **bistream** format is what enables **streaming inference** — the model can start producing speech tokens before all text tokens are available. The `FILL` token signals the model to pause speech generation and accept the next batch of text tokens.

### Instruct Support (New in CosyVoice 3)
CosyVoice 3 adds an **instruct token** sequence before the text tokens. This allows controlling language, dialect, emotion, speed, etc. via a system prompt like:
```
You are a helpful assistant. Please speak in Cantonese.<|endofprompt|>
```
The `<|endofprompt|>` (token ID 151646) separates the system instruction from the actual text to be spoken.

In [ ]:
# File: cosyvoice/llm/llm.py — Training forward pass (simplified)
# This shows how the bistream/unistream sequences are constructed

def prepare_lm_input_target(self, sos_emb, text_token, text_token_emb, text_token_len,
                             task_id_emb, speech_token, speech_token_emb, speech_token_len,
                             instruct_token=None, instruct_token_emb=None, instruct_token_len=None):
    for i in range(batch_size):
        # 50% chance: use bistream format (if speech/text ratio permits)
        if random.random() < 0.5 and speech_token_len[i] / text_token_len[i] > 15 / 5:
            # Bistream: interleave 5 text tokens with 15 speech tokens
            lm_target = [IGNORE_ID]  # SOS has no target
            lm_input  = [sos_emb]
            # prepend instruct tokens (targets ignored)
            lm_target += [IGNORE_ID] * instruct_token_len[i]
            lm_input.append(instruct_token_emb[i])
            
            for j in range(ceil(text_token_len / 5)):
                this_text   = text_token[i][j*5 : (j+1)*5]    # 5 text tokens
                this_speech = speech_token[i][j*15 : (j+1)*15] # 15 speech tokens
                if len(this_text) == 5:
                    # Target: ignore text prediction, predict speech + FILL
                    lm_target += [IGNORE_ID] * 4 + this_speech.tolist() + [FILL_TOKEN]
                    lm_input.append(text_token_emb[i][...])
                    lm_input.append(speech_token_emb[i][...])
                else:
                    # Last chunk: remaining text + TASK_ID + remaining speech + EOS
                    lm_target += [IGNORE_ID]*len(this_text) + remaining_speech + [EOS]
                    lm_input.append(remaining_text_emb + task_id_emb + remaining_speech_emb)
        
        # 50% chance: use unistream format
        else:
            lm_target = [IGNORE_ID] * (1 + instruct_len + text_len) + speech_tokens + [EOS]
            lm_input  = concat([sos_emb, instruct_emb, text_emb, task_id_emb, speech_emb])

## 2.3 LLM Inference Flow

During inference, the LLM generates speech tokens one at a time using **KV-cache** for efficient autoregressive decoding. It supports two modes:

1. **Unistream inference** (`inference`): All text is available upfront
2. **Bistream inference** (`inference_bistream`): Text arrives as a generator (e.g., from an LLM), speech tokens are produced progressively

The sampling strategy is **Repetition-Aware Sampling (RAS)** with `top_p=0.8, top_k=25`, which penalizes recently-generated tokens to avoid repetition artifacts.

In [ ]:
# File: cosyvoice/llm/llm.py — Inference (unistream, simplified)

@torch.inference_mode()
def inference(self, text, text_len, prompt_text, prompt_text_len,
              prompt_speech_token, prompt_speech_token_len, embedding,
              sampling=25, max_token_text_ratio=20, min_token_text_ratio=2):
    
    # 1. Concatenate prompt_text + synthesis text, embed via Qwen2
    text = torch.concat([prompt_text, text], dim=1)
    text_emb = self.llm.model.model.embed_tokens(text)  # Qwen2 embedding layer
    
    # 2. Build initial LLM input sequence:
    #    [SOS] [text_embeddings] [TASK_ID] [prompt_speech_token_embeddings]
    sos_emb = self.speech_embedding.weight[self.sos].reshape(1, 1, -1)
    task_id_emb = self.speech_embedding.weight[self.task_id].reshape(1, 1, -1)
    prompt_speech_token_emb = self.speech_embedding(prompt_speech_token)
    lm_input = torch.concat([sos_emb, text_emb, task_id_emb, prompt_speech_token_emb], dim=1)
    
    # 3. Autoregressive decoding with KV-cache
    min_len = int((text_len - prompt_text_len) * min_token_text_ratio)
    max_len = int((text_len - prompt_text_len) * max_token_text_ratio)
    
    out_tokens, cache = [], None
    for i in range(max_len):
        y_pred, cache = self.llm.forward_one_step(lm_input, masks=..., cache=cache)
        logp = self.llm_decoder(y_pred[:, -1]).log_softmax(dim=-1)
        
        # RAS sampling (avoids repetition)
        top_ids = self.sampling_ids(logp, out_tokens, sampling,
                                     ignore_eos=(i < min_len))
        if top_ids in self.stop_token_ids:
            break
        yield top_ids  # Stream tokens one by one
        out_tokens.append(top_ids)
        lm_input = self.speech_embedding.weight[top_ids].reshape(1, 1, -1)

---

# 3. Stage 2: Flow Matching with DiT (Speech Token -> Mel Spectrogram)

## 3.1 Overview

The flow matching module converts discrete speech tokens into continuous **80-dimensional mel spectrograms**. CosyVoice 3 introduces a major architectural upgrade here: replacing the Conformer-based UNet decoder used in v2 with a **Diffusion Transformer (DiT)**.

The flow module class is `CausalMaskedDiffWithDiT` and consists of three sub-components:

| Sub-component | Class | Role |
|---|---|---|
| **Token Embedding + PreLookahead** | `nn.Embedding` + `PreLookaheadLayer` | Embeds speech tokens and applies causal convolution with limited lookahead |
| **Flow Matching Decoder** | `CausalConditionalCFM` | Conditional Flow Matching ODE solver |
| **DiT Estimator** | `DiT` | Transformer-based velocity field estimator (22 layers, dim=1024) |

### Key Parameters (from `cosyvoice3.yaml`)
- `input_size / output_size`: 80 (mel dimension)
- `vocab_size`: 6561 (FSQ speech tokens)
- `token_mel_ratio`: 2 (each speech token corresponds to 2 mel frames)
- `pre_lookahead_len`: 3 (limited future context for streaming)
- `token_frame_rate`: 25 Hz (25 speech tokens per second of audio)
- Sample rate: **24000 Hz**

In [ ]:
# File: cosyvoice/flow/flow.py — CausalMaskedDiffWithDiT (CosyVoice 3 flow module)

class CausalMaskedDiffWithDiT(torch.nn.Module):
    def __init__(self, input_size=80, output_size=80, spk_embed_dim=192,
                 vocab_size=6561, input_frame_rate=25, token_mel_ratio=2,
                 pre_lookahead_len=3, pre_lookahead_layer=None, decoder=None):
        
        # Embed speech tokens (6561 vocab) -> 80-dim vectors
        self.input_embedding = nn.Embedding(vocab_size, input_size)  # 6561 -> 80
        
        # Speaker embedding projection: 192-dim x-vector -> 80-dim
        self.spk_embed_affine_layer = nn.Linear(spk_embed_dim, output_size)
        
        # PreLookaheadLayer: causal convolution allowing 3 tokens of future context
        # This enables streaming while maintaining quality
        self.pre_lookahead_layer = pre_lookahead_layer  # Conv1d-based
        
        # token_mel_ratio=2: each token maps to 2 mel frames
        # So for 25 tokens/sec -> 50 mel frames/sec -> at hop_size=480, sample_rate=24000
        self.token_mel_ratio = token_mel_ratio
        
        # The main decoder: CausalConditionalCFM (flow matching)
        self.decoder = decoder  # Contains the DiT estimator inside

## 3.2 The PreLookaheadLayer

A key innovation for **streaming support**. This is a simple 2-layer causal Conv1d that allows the model to peek at `pre_lookahead_len=3` future tokens during streaming inference, then applies a residual connection.

During training, padding is used. During streaming inference, the actual future context tokens are passed explicitly.

In [ ]:
# File: cosyvoice/transformer/upsample_encoder.py — PreLookaheadLayer

class PreLookaheadLayer(nn.Module):
    """Causal convolution with limited lookahead for streaming inference."""
    
    def __init__(self, in_channels=80, channels=1024, pre_lookahead_len=3):
        self.pre_lookahead_len = pre_lookahead_len
        # First conv: kernel_size = pre_lookahead_len + 1 = 4
        # This allows looking 3 tokens into the future
        self.conv1 = nn.Conv1d(in_channels, channels, kernel_size=4, stride=1, padding=0)
        # Second conv: kernel_size = 3, causal (left-padded)
        self.conv2 = nn.Conv1d(channels, in_channels, kernel_size=3, stride=1, padding=0)

    def forward(self, inputs, context=torch.zeros(0, 0, 0)):
        outputs = inputs.transpose(1, 2)  # (B, T, C) -> (B, C, T)
        
        if context.size(2) == 0:
            # Training: zero-pad future positions
            outputs = F.pad(outputs, (0, self.pre_lookahead_len))
        else:
            # Streaming inference: use actual future context
            outputs = torch.concat([outputs, context.transpose(1, 2)], dim=2)
        
        outputs = F.leaky_relu(self.conv1(outputs))
        outputs = F.pad(outputs, (2, 0))  # causal padding for conv2
        outputs = self.conv2(outputs).transpose(1, 2)
        
        return outputs + inputs  # residual connection

## 3.3 The DiT (Diffusion Transformer) Estimator

The DiT is the velocity field estimator used inside the flow matching ODE. It replaces the UNet-style decoder used in CosyVoice 1/2.

**Architecture details** (from `cosyvoice3.yaml`):
- **22 DiT blocks** (depth=22)
- **Hidden dim**: 1024
- **Attention**: 16 heads, 64 dim per head
- **Feed-forward**: 2x multiplier (ff_mult=2)
- **Positional encoding**: Rotary Position Embedding (RoPE)
- **Causal attention** with chunk masking for streaming (static_chunk_size=50 mel frames = 25 tokens)
- **Classifier-Free Guidance (CFG)** during inference (cfg_rate=0.7)

The DiT takes as input the concatenation of:
1. **x**: noised mel spectrogram (being denoised)
2. **cond**: condition (prompt mel spectrogram + zeros for target region)
3. **mu**: encoder output (upsampled speech token representations)
4. **spks**: speaker embedding (80-dim, broadcast across time)
5. **t**: diffusion timestep embedding

In [ ]:
# File: cosyvoice/flow/DiT/dit.py — The DiT model

class DiT(nn.Module):
    def __init__(self, dim=1024, depth=22, heads=16, dim_head=64,
                 ff_mult=2, mel_dim=80, mu_dim=80, spk_dim=80,
                 out_channels=80, static_chunk_size=50, num_decoding_left_chunks=-1):
        
        # Timestep embedding: scalar t -> dim-dimensional vector
        self.time_embed = TimestepEmbedding(dim)  # sinusoidal -> MLP
        
        # Input embedding: concatenates [x, cond, mu, spks] and projects
        # Input: mel_dim*2 + mu_dim + spk_dim = 80*2 + 80 + 80 = 320 -> 1024
        self.input_embed = InputEmbedding(mel_dim, mu_dim, dim, spk_dim)
        
        # Rotary position embedding for attention
        self.rotary_embed = RotaryEmbedding(dim_head)  # 64-dim
        
        # 22 DiT blocks, each with:
        #   - AdaLayerNorm (conditioned on timestep t)
        #   - Multi-head self-attention (16 heads, 64 dim/head)
        #   - Feed-forward network (dim * ff_mult = 1024 * 2 = 2048)
        self.transformer_blocks = nn.ModuleList([
            DiTBlock(dim=1024, heads=16, dim_head=64, ff_mult=2)
            for _ in range(22)
        ])
        
        # Final projection: 1024 -> 80 (mel spectrogram)
        self.norm_out = AdaLayerNormZero_Final(dim)
        self.proj_out = nn.Linear(dim, mel_dim)  # 1024 -> 80
        
        # Streaming: chunk-based causal attention
        self.static_chunk_size = static_chunk_size  # 50 mel frames

    def forward(self, x, mask, mu, t, spks=None, cond=None, streaming=False):
        # x, mu, cond: (B, 80, T) -> transpose to (B, T, 80)
        t = self.time_embed(t)                         # (B,) -> (B, 1024)
        x = self.input_embed(x, cond, mu, spks)        # (B, T, 1024)
        rope = self.rotary_embed.forward_from_seq_len(seq_len)
        
        # Causal chunk mask for streaming, full attention for non-streaming
        if streaming:
            attn_mask = chunk_mask(static_chunk_size=50)
        else:
            attn_mask = full_attention_mask()
        
        for block in self.transformer_blocks:           # 22 DiT blocks
            x = block(x, t, mask=attn_mask, rope=rope)
        
        x = self.norm_out(x, t)
        return self.proj_out(x).transpose(1, 2)        # (B, 80, T)

## 3.4 Flow Matching: Training and Inference

### Background: What is Flow Matching?

Flow matching is a generative modeling technique. The core idea is to learn a **velocity field** $v_\theta(x, t)$ that describes how to continuously transform a simple distribution (Gaussian noise at $t=0$) into a complex target distribution (mel spectrograms at $t=1$). This is formalized as an Ordinary Differential Equation (ODE):

$$\frac{dx}{dt} = v_\theta(x, t), \quad t \in [0, 1]$$

At inference, we start from noise $x_0 \sim \mathcal{N}(0, I)$ and integrate this ODE forward to $t=1$ to produce the mel spectrogram.

### Training: Conditional Flow Matching (CFM) Loss

The training objective teaches the DiT estimator to predict the velocity field at any point along the interpolation path. Here is how it works step by step.

**Step 1 — Sample a random timestep.** For each sample in the batch, draw $t$ uniformly:

$$t \sim \text{Uniform}(0, 1) \quad \text{shape: } (B, 1, 1)$$

**Step 2 — Sample noise.** Draw noise $z$ from the standard Gaussian, same shape as the target mel $x_1$:

$$z \sim \mathcal{N}(0, I) \quad \text{shape: } (B, 80, T_{\text{mel}})$$

**Step 3 — Compute the interpolated sample $y$.** This is the point along the straight-line path between noise and target at time $t$. Here $\sigma_{\min} = 10^{-6}$ is a tiny constant that prevents the variance from collapsing to exactly zero:

$$y = \underbrace{(1 - (1 - \sigma_{\min}) \cdot t)}_{\text{weight on noise, decreases as } t \to 1} \cdot z + \underbrace{t}_{\text{weight on target, increases as } t \to 1} \cdot x_1$$

When $t=0$: $y \approx z$ (pure noise). When $t=1$: $y \approx x_1$ (pure target).

**Step 4 — Compute the ground-truth velocity $u$.** This is the derivative of $y$ with respect to $t$ — the direction the flow should push:

$$u = \frac{\partial y}{\partial t} = x_1 - (1 - \sigma_{\min}) \cdot z$$

This is a constant vector (independent of $t$) pointing from the noise toward the target.

**Step 5 — Classifier-Free Guidance (CFG) dropout during training.** With probability `training_cfg_rate=0.2` (20%), the conditioning inputs ($\mu$, speaker embedding, mel condition) are **zeroed out**. This trains the model to generate both conditionally and unconditionally, enabling CFG at inference:

$$\mu' = \mu \cdot m, \quad \text{spk}' = \text{spk} \cdot m, \quad \text{cond}' = \text{cond} \cdot m$$
$$\text{where } m_i \sim \text{Bernoulli}(0.8) \text{ per sample in batch}$$

**Step 6 — Predict and compute loss.** Feed the interpolated sample $y$ along with conditioning to the DiT, and minimize MSE against the true velocity $u$:

$$\hat{u} = \text{DiT}(y, \; \text{mask}, \; \mu', \; t, \; \text{spk}', \; \text{cond}')$$

$$\mathcal{L} = \frac{\sum \| (\hat{u} - u) \odot \text{mask} \|^2}{\sum \text{mask} \times 80}$$

The mask ensures padding frames do not contribute. The normalization is by (number of valid frames $\times$ 80 mel channels).

---

### Inference: Euler ODE Solver with CFG

At inference, we solve the ODE from $t=0$ (noise) to $t=1$ (mel) using **10 Euler steps** with a **cosine time schedule**.

**Cosine time schedule** — instead of uniform spacing, the time steps are cosine-warped so that more steps concentrate near $t=0$ (where the signal-to-noise ratio is low and accuracy matters most):

$$t_{\text{span}} = 1 - \cos\!\left(\text{linspace}(0, 1, 11) \times \frac{\pi}{2}\right)$$

This produces: $[0.0, 0.024, 0.095, 0.206, 0.345, 0.5, 0.655, 0.794, 0.905, 0.976, 1.0]$ — denser at the start.

**Classifier-Free Guidance at inference** — at each Euler step, the estimator is run **twice** in a single batched forward pass (batch dim = 2): once with full conditioning (row 0), once with all conditions zeroed out (row 1). The two predictions are combined with guidance rate $\gamma = 0.7$:

$$\hat{v} = (1 + \gamma) \cdot v_{\text{cond}} - \gamma \cdot v_{\text{uncond}}$$

This amplifies the conditioned prediction while subtracting the unconditional baseline, steering generation more strongly toward the target speaker/content.

**CausalConditionalCFM's fixed noise** — for CosyVoice 3 streaming inference, `CausalConditionalCFM` pre-generates a **fixed random noise** tensor (seeded with 0, covering up to ~5 min of audio) at initialization. This ensures that when processing overlapping chunks, the noise for previously generated frames stays identical, preventing artifacts at chunk boundaries.

In [ ]:
# File: cosyvoice/flow/flow_matching.py
# Complete training loss + inference with all variables defined and annotated

# ============================================================
# PART 1: Training Loss — ConditionalCFM.compute_loss()
# ============================================================

def compute_loss(self, x1, mask, mu, spks=None, cond=None, streaming=False):
    """
    Conditional Flow Matching training loss.

    Args:
        x1:   Target mel spectrogram, shape (B, 80, T_mel)
        mask: Valid frame mask,        shape (B, 1, T_mel)   — 1 for real frames, 0 for padding
        mu:   Encoder output (conditioning), shape (B, 80, T_mel) — from PreLookaheadLayer
        spks: Speaker embedding,       shape (B, 80)         — projected from 192-dim to 80-dim
        cond: Prompt mel condition,     shape (B, 80, T_mel)  — first N frames copied from reference audio

    Returns:
        loss: scalar, the MSE flow matching loss
        y:    the interpolated sample (for debugging/logging)
    """
    b, _, t = mu.shape  # b = batch_size, t is overwritten below

    # Step 1: Sample a random timestep t ~ Uniform(0, 1) for each sample
    t = torch.rand([b, 1, 1], device=mu.device, dtype=mu.dtype)  # shape: (B, 1, 1)

    # Step 2: Sample noise z ~ N(0, I), same shape as target mel x1
    z = torch.randn_like(x1)  # shape: (B, 80, T_mel)

    # Step 3: Interpolate between noise (z) and target (x1) at timestep t
    #   y(t) = (1 - (1-σ_min)*t) * z + t * x1
    #   At t=0: y ≈ z (noise).  At t=1: y ≈ x1 (target).
    sigma_min = 1e-6  # self.sigma_min from config
    y = (1 - (1 - sigma_min) * t) * z + t * x1  # shape: (B, 80, T_mel)

    # Step 4: Ground-truth velocity u = dy/dt = x1 - (1-σ_min)*z
    #   This is the direction the model should learn to predict
    u = x1 - (1 - sigma_min) * z  # shape: (B, 80, T_mel)

    # Step 5: Classifier-Free Guidance dropout (training_cfg_rate = 0.2)
    #   With 20% probability, zero out ALL conditioning for this sample
    #   so the model also learns unconditional generation
    if self.training_cfg_rate > 0:
        cfg_mask = torch.rand(b, device=x1.device) > self.training_cfg_rate  # True = keep, False = drop
        mu   = mu   * cfg_mask.view(-1, 1, 1)  # zero out encoder output
        spks = spks * cfg_mask.view(-1, 1)      # zero out speaker embedding
        cond = cond * cfg_mask.view(-1, 1, 1)   # zero out prompt mel condition

    # Step 6: DiT predicts velocity at interpolated point y
    pred = self.estimator(y, mask, mu, t.squeeze(), spks, cond, streaming=streaming)
    # pred shape: (B, 80, T_mel)

    # Step 7: MSE loss between predicted and true velocity, masked to valid frames
    loss = F.mse_loss(pred * mask, u * mask, reduction="sum") / (torch.sum(mask) * u.shape[1])
    #   numerator: sum of squared errors over all (batch, channel, time) where mask=1
    #   denominator: (total valid frames across batch) × 80 mel channels
    return loss, y


# ============================================================
# PART 2: Inference — CausalConditionalCFM (CosyVoice 3)
# ============================================================

class CausalConditionalCFM(ConditionalCFM):
    def __init__(self, in_channels, cfm_params, n_spks=1, spk_emb_dim=64, estimator=None):
        super().__init__(in_channels, cfm_params, n_spks, spk_emb_dim, estimator)
        # Pre-generate fixed noise at seed=0 for reproducible streaming.
        # Covers up to 50*300 = 15000 mel frames ≈ 5 minutes at 50 fps.
        set_all_random_seed(0)
        self.rand_noise = torch.randn([1, 80, 50 * 300])  # shape: (1, 80, 15000)

    @torch.inference_mode()
    def forward(self, mu, mask, n_timesteps=10, temperature=1.0,
                spks=None, cond=None, streaming=False):
        """
        Args:
            mu:   Encoder output,  shape (1, 80, T_mel)
            mask: Valid mask,      shape (1, 1, T_mel)
            n_timesteps: ODE solver steps (default 10)
            spks: Speaker embed,   shape (1, 80)
            cond: Prompt mel,      shape (1, 80, T_mel)
        """
        # Use pre-generated noise (sliced to current length) for streaming consistency
        T_mel = mu.size(2)
        z = self.rand_noise[:, :, :T_mel].to(mu.device).to(mu.dtype) * temperature

        # Build cosine time schedule: 11 points from t=0 to t=1
        t_span = torch.linspace(0, 1, n_timesteps + 1, device=mu.device, dtype=mu.dtype)
        t_span = 1 - torch.cos(t_span * 0.5 * torch.pi)  # cosine warp
        # Result: [0.0, 0.024, 0.095, 0.206, 0.345, 0.5, 0.655, 0.794, 0.905, 0.976, 1.0]

        return self.solve_euler(z, t_span, mu, mask, spks, cond, streaming), None

    def solve_euler(self, x, t_span, mu, mask, spks, cond, streaming=False):
        """
        Euler ODE solver with Classifier-Free Guidance.

        Args:
            x:      Starting noise,          shape (1, 80, T_mel)
            t_span: Time schedule,            shape (11,)  — 10 steps + start
            mu:     Encoder conditioning,     shape (1, 80, T_mel)
            mask:   Valid frame mask,          shape (1, 1, T_mel)
            spks:   Speaker embedding,         shape (1, 80)
            cond:   Prompt mel condition,       shape (1, 80, T_mel)
        """
        t = t_span[0].unsqueeze(0)       # current time, shape (1,)
        dt = t_span[1] - t_span[0]       # first step size

        # Pre-allocate batch=2 tensors: row 0 = conditioned, row 1 = unconditioned (zeros)
        # Using pre-allocated tensors avoids torch.cat/stack which can cause
        # memory format issues with TensorRT inference
        x_in    = torch.zeros([2, 80, x.size(2)], device=x.device, dtype=spks.dtype)
        mask_in = torch.zeros([2, 1,  x.size(2)], device=x.device, dtype=spks.dtype)
        mu_in   = torch.zeros([2, 80, x.size(2)], device=x.device, dtype=spks.dtype)
        t_in    = torch.zeros([2],                 device=x.device, dtype=spks.dtype)
        spks_in = torch.zeros([2, 80],             device=x.device, dtype=spks.dtype)
        cond_in = torch.zeros([2, 80, x.size(2)], device=x.device, dtype=spks.dtype)
        # Row 1 stays all-zeros throughout = unconditional input for CFG

        for step in range(1, len(t_span)):  # 10 iterations
            # Fill row 0 (conditioned) and row 1 (unconditioned = zeros already)
            x_in[:]  = x          # both rows get the current state x
            mask_in[:] = mask     # both rows get the same mask
            mu_in[0]   = mu       # row 0: real encoder output;  row 1: zeros
            t_in[:]    = t        # both rows get the same timestep
            spks_in[0] = spks     # row 0: real speaker embed;   row 1: zeros
            cond_in[0] = cond     # row 0: real mel condition;   row 1: zeros

            # Single batched forward pass through DiT — processes both conditions at once
            dphi_dt = self.estimator(x_in, mask_in, mu_in, t_in, spks_in, cond_in,
                                     streaming=streaming)
            # dphi_dt shape: (2, 80, T_mel)

            # Split into conditioned (row 0) and unconditioned (row 1) predictions
            dphi_cond, dphi_uncond = dphi_dt.chunk(2)  # each (1, 80, T_mel)

            # Apply CFG: v_hat = (1 + γ) * v_cond - γ * v_uncond,  γ = 0.7
            dphi_dt = (1.0 + self.inference_cfg_rate) * dphi_cond \
                     - self.inference_cfg_rate * dphi_uncond

            # Euler step: x_{t+dt} = x_t + dt * v(x_t, t)
            x = x + dt * dphi_dt
            t = t + dt

            # Update dt for next step (non-uniform due to cosine schedule)
            if step < len(t_span) - 1:
                dt = t_span[step + 1] - t

        return x.float()  # final mel spectrogram, shape (1, 80, T_mel)

---

# 4. Stage 3: HiFi-GAN Vocoder (Mel -> Waveform)

The vocoder converts 80-dim mel spectrograms into 24kHz audio waveforms. CosyVoice 3 uses `CausalHiFTGenerator`, a **causal** variant of the HiFi-GAN vocoder with:

- **Source-filter architecture**: A neural source module (sine generator) produces harmonic excitation based on predicted F0, which is then filtered by a stack of transposed convolutions
- **Causal convolutions** throughout (for streaming compatibility)
- **F0 predictor**: `CausalConvRNNF0Predictor` estimates pitch from the mel spectrogram
- **iSTFT head**: Final waveform synthesis uses inverse STFT (n_fft=16, hop_len=4) for efficiency
- **Snake activation**: Used instead of LeakyReLU in residual blocks

### Key parameters:
- Upsampling rates: [8, 5, 3] (total upsampling = 8 * 5 * 3 * 4 = 480 = hop_size)
- Base channels: 512
- 8 harmonics for the sine generator
- Sample rate: 24000 Hz

The vocoder is trained separately as a GAN with multi-period and multi-resolution spectral discriminators.

In [ ]:
# File: cosyvoice/hifigan/generator.py — CausalHiFTGenerator (simplified structure)
# From cosyvoice3.yaml configuration

hift_config = {
    "in_channels": 80,           # mel spectrogram dimensions
    "base_channels": 512,
    "nb_harmonics": 8,           # number of harmonic overtones
    "sampling_rate": 24000,
    "nsf_alpha": 0.1,
    "nsf_sigma": 0.003,
    "nsf_voiced_threshold": 10,
    "upsample_rates": [8, 5, 3],          # total: 8*5*3 = 120, * istft_hop=4 -> 480 = hop_size
    "upsample_kernel_sizes": [16, 11, 7],
    "istft_params": {"n_fft": 16, "hop_len": 4},
    "resblock_kernel_sizes": [3, 7, 11],
    "resblock_dilation_sizes": [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
    "audio_limit": 0.99,
    "conv_pre_look_right": 4,    # causal conv lookahead
    "f0_predictor": "CausalConvRNNF0Predictor(in_channels=80, cond_channels=512)",
}

# The generator pipeline:
#   mel (B, 80, T)
#     -> F0 predictor -> f0 (B, 1, T)
#     -> SineGen2(f0) -> harmonic source (B, T*480, 9)  [8 harmonics + fundamental]
#     -> SourceModule (conv) -> source signal
#     -> Upsample blocks (ConvTranspose1d x3: 8x, 5x, 3x)
#     -> ResBlocks with Snake activation
#     -> iSTFT synthesis -> waveform (B, 1, T*480)
print("Total upsampling factor:", 8 * 5 * 3 * 4, "== hop_size 480")
print("Mel frame rate:", 24000 / 480, "= 50 fps")
print("Token frame rate: 25 Hz, token_mel_ratio=2, so 25*2 = 50 mel fps")

---

# 5. The Tokenizer and Frontend

## 5.1 Text Tokenizer

CosyVoice 3 uses a **Qwen2-based tokenizer** (`CosyVoice3Tokenizer`) with extensive special tokens:

- **Qwen2 base vocabulary** (standard BPE tokens for multilingual text)
- **Paralinguistic tokens**: `[breath]`, `[laughter]`, `[cough]`, `[noise]`, `[sigh]`, etc.
- **Control tokens**: `<strong>`, `</strong>`, `<|endofprompt|>`, `<|endofsystem|>`
- **CMU Phoneme tokens** (English): `[AA]`, `[AE]`, `[B]`, `[CH]`, ... (for pronunciation inpainting)
- **Chinese Pinyin tokens**: `[a]`, `[ai]`, `[an]`, `[ang]`, `[b]`, `[c]`, ... with tones (`[ā]`, `[á]`, `[ǎ]`, `[à]`)

This allows fine-grained pronunciation control — users can embed phoneme tokens directly in the text to force specific pronunciations.

## 5.2 Speech Tokenizer

The speech tokenizer converts raw audio into discrete tokens at **25 Hz** using a **Finite Scalar Quantization (FSQ)** model:
- Vocabulary: **6561 tokens** (= 3^8, suggesting 8 codebook levels with 3 values each, or a similar FSQ configuration)
- ONNX model: `speech_tokenizer_v3.onnx`
- Operates on Whisper-style features

## 5.3 Speaker Embedding

Speaker identity is captured using **CAM++** (an ONNX model `campplus.onnx`), producing **192-dimensional speaker embeddings** (x-vectors). These are L2-normalized and projected to the required dimensions for each component.

In [ ]:
# File: cosyvoice/tokenizer/tokenizer.py — CosyVoice3Tokenizer (key excerpts)

class CosyVoice3Tokenizer(CosyVoice2Tokenizer):
    """Qwen2-based tokenizer with pronunciation inpainting support."""
    
    def __init__(self, token_path, skip_special_tokens=True):
        special_tokens = {
            'eos_token': '<|endoftext|>',
            'pad_token': '<|endoftext|>',
            'additional_special_tokens': [
                # System/control tokens
                '<|im_start|>', '<|im_end|>', '<|endofprompt|>', '<|endofsystem|>',
                
                # Paralinguistic tokens
                '[breath]', '[laughter]', '[cough]', '[noise]',
                '[sigh]', '[lipsmack]', '[hissing]',
                '<strong>', '</strong>',
                
                # English CMU phonemes (examples)
                '[AA]', '[AA0]', '[AA1]', '[AA2]',  # as in "odd"
                '[AE]', '[AE0]', '[AE1]', '[AE2]',  # as in "at"
                '[B]', '[CH]', '[D]', '[F]', '[G]',  # consonants
                # ... 80+ phoneme tokens total
                
                # Chinese Pinyin with tones (examples)
                '[a]', '[ai]', '[an]', '[ang]', '[ao]',
                '[ā]', '[á]', '[ǎ]', '[à]',           # toned vowels
                '[b]', '[c]', '[ch]', '[d]', '[f]',    # initials
                # ... 200+ pinyin tokens total
            ]
        }
        self.tokenizer = AutoTokenizer.from_pretrained(token_path)
        self.tokenizer.add_special_tokens(special_tokens)

# Example: pronunciation inpainting in CosyVoice 3
# The user can embed phonemes directly in text to override pronunciation:
example_text = '高管也通过电话、短信、微信等方式对报道[j][ǐ]予好评。'
# Here [j][ǐ] forces the pronunciation of 给 as "jǐ" instead of default "gěi"

---

# 6. End-to-End Inference Pipeline

Here is the complete inference flow, showing how the three stages connect. The LLM runs in a **separate thread** so speech token generation and mel/waveform synthesis can overlap for streaming.

In [ ]:
# File: cosyvoice/cli/model.py — CosyVoice3Model.tts() (simplified)

class CosyVoice3Model(CosyVoice2Model):
    """
    Orchestrates the 3-stage TTS pipeline with streaming support.
    
    token_hop_len = 25 tokens per chunk (1 second of audio)
    stream_scale_factor = 2 (chunk size doubles progressively)
    """
    
    # FSQ silent and breath tokens (filtered during generation to avoid excessive pauses)
    silent_tokens = [1, 2, 28, 29, 55, 248, 494, 2241, 2242, 2322, 2323]
    
    def tts(self, text, flow_embedding, llm_embedding, prompt_text,
            llm_prompt_speech_token, flow_prompt_speech_token,
            prompt_speech_feat, stream=False, speed=1.0):
        
        this_uuid = str(uuid.uuid1())
        
        # Stage 1: LLM generates speech tokens in a BACKGROUND THREAD
        # This allows overlap with Stage 2+3
        p = threading.Thread(target=self.llm_job,
                             args=(text, prompt_text, llm_prompt_speech_token,
                                   llm_embedding, this_uuid))
        p.start()
        
        if stream:
            token_offset = 0
            while True:
                time.sleep(0.1)  # poll for new tokens
                
                # When enough tokens accumulated, run flow + vocoder on chunk
                if len(self.tts_speech_token_dict[this_uuid]) - token_offset >= token_hop_len:
                    token_chunk = self.tts_speech_token_dict[this_uuid][:offset + hop_len]
                    
                    # Stage 2+3: token -> mel -> waveform
                    tts_speech = self.token2wav(
                        token=token_chunk,
                        prompt_token=flow_prompt_speech_token,
                        prompt_feat=prompt_speech_feat,
                        embedding=flow_embedding,
                        token_offset=token_offset,
                        stream=True, finalize=False
                    )
                    yield {'tts_speech': tts_speech.cpu()}
                    token_offset += token_hop_len
        else:
            p.join()  # wait for all tokens
            all_tokens = self.tts_speech_token_dict[this_uuid]
            tts_speech = self.token2wav(all_tokens, ..., finalize=True, speed=speed)
            yield {'tts_speech': tts_speech.cpu()}

In [ ]:
# File: cosyvoice/cli/model.py — CosyVoice3Model.token2wav()
# This is the Stage 2+3 method: speech tokens -> mel -> waveform

def token2wav(self, token, prompt_token, prompt_feat, embedding,
              token_offset, uuid, stream=False, finalize=False, speed=1.0):
    """
    Convert speech tokens to waveform via Flow Matching + HiFi-GAN.
    
    In streaming mode, mel is accumulated in a cache and the vocoder
    only synthesizes newly generated mel frames (avoiding redundant computation).
    """
    with torch.cuda.amp.autocast(self.fp16):
        # Stage 2: Flow Matching — speech tokens -> mel spectrogram
        tts_mel, _ = self.flow.inference(
            token=token,                    # (1, T_token)
            prompt_token=prompt_token,      # (1, T_prompt_token) 
            prompt_feat=prompt_feat,        # (1, T_prompt_mel, 80)
            embedding=embedding,            # (1, 192) speaker embedding
            streaming=stream,
            finalize=finalize
        )
        # Only take newly generated mel frames (skip already-synthesized ones)
        tts_mel = tts_mel[:, :, token_offset * self.flow.token_mel_ratio:]
        
        # Mel caching for streaming: accumulate mel and track speech offset
        if self.hift_cache_dict[uuid] is not None:
            tts_mel = torch.concat([cached_mel, tts_mel], dim=2)
        self.hift_cache_dict[uuid] = {'mel': tts_mel, 'speech_offset': offset}
        
        # Speed control (non-streaming only): interpolate mel length
        if speed != 1.0:
            tts_mel = F.interpolate(tts_mel, size=int(tts_mel.shape[2] / speed))
        
        # Stage 3: HiFi-GAN Vocoder — mel -> waveform
        tts_speech, _ = self.hift.inference(speech_feat=tts_mel, finalize=finalize)
        
        # Only return newly synthesized audio (skip cached portion)
        tts_speech = tts_speech[:, self.hift_cache_dict[uuid]['speech_offset']:]
    
    return tts_speech

---

# 7. Training Pipeline

CosyVoice 3 has a **modular training pipeline** where each of the three components (LLM, Flow, HiFi-GAN) is trained independently. The training script is `examples/libritts/cosyvoice3/run.sh`.

## 7.1 Data Preparation Pipeline

```
Stage 0: prepare_data.py    → wav.scp, text, utt2spk, spk2utt (with instruct prefix)
Stage 1: extract_embedding   → speaker embeddings via CAM++ (campplus.onnx)
Stage 2: extract_speech_token → discrete speech tokens via speech_tokenizer_v3.onnx
Stage 3: make_parquet_list   → Parquet format for efficient data loading
```

Note: Stages 1 and 2 (embedding/token extraction) are **optional** — the system supports **online feature extraction** during training, though this is slower.

In [ ]:
# File: examples/libritts/cosyvoice3/run.sh — Data preparation (excerpt)

# Stage 0: Prepare data with instruct prefix for CosyVoice3
# NOTE: CosyVoice 3 adds an instruct string to each utterance
"""
python local/prepare_data.py \
    --src_dir $data_dir/LibriTTS/$x \
    --des_dir data/$x \
    --instruct "You are a helpful assistant.<|endofprompt|>"
"""

# Stage 1: Extract speaker embeddings (CAM++ model)
"""
tools/extract_embedding.py --dir data/$x \
    --onnx_path $pretrained_model_dir/campplus.onnx
"""

# Stage 2: Extract discrete speech tokens (FSQ tokenizer v3)
"""
tools/extract_speech_token.py --dir data/$x \
    --onnx_path $pretrained_model_dir/speech_tokenizer_v3.onnx
"""

# Stage 3: Convert to Parquet format (1000 utterances per file)
"""
tools/make_parquet_list.py --num_utts_per_parquet 1000 \
    --num_processes 10 \
    --src_dir data/$x \
    --des_dir data/$x/parquet
"""

## 7.2 Data Processing Pipeline

The data processing pipeline is defined in `cosyvoice3.yaml` and consists of these sequential stages:

In [ ]:
# File: cosyvoice3.yaml — data pipeline configuration

data_pipeline = [
    "parquet_opener",        # Read Parquet files, iterate over rows
    "tokenize",              # Tokenize text using CosyVoice3Tokenizer (Qwen2-based)
    "filter",                # Filter: 100 < num_frames < 6000, 1 < token_len < 200
    "resample",              # Resample audio to 24000 Hz
    "compute_fbank",         # Extract 80-dim mel spectrogram (n_fft=1920, hop=480)
    "parse_embedding",       # Load and L2-normalize speaker embeddings
    "compute_whisper_fbank", # Compute Whisper-style features (for online speech tokenizer)
    "shuffle",               # Shuffle with buffer_size=1000
    "sort",                  # Sort by length within buffer (sort_size=500)
    "batch",                 # Dynamic batching: max_frames_in_batch=2000
    "padding",               # Pad sequences to batch max length
]

# For GAN (vocoder) training, the pipeline adds:
data_pipeline_gan = [
    # ... same as above, but also includes:
    "truncate",              # Truncate to 24960 samples (must be multiple of hop_size * token_mel_ratio)
    "compute_f0",            # Extract F0 using pyworld (needed for source-filter vocoder)
]

## 7.3 Training Each Component

All three models are trained via the same `cosyvoice/bin/train.py` entry point with `--model` flag selecting which component to train. Training uses **torch DDP** or **DeepSpeed** for distributed training.

### LLM Training
- **Loss**: Label Smoothing Cross-Entropy on speech token prediction (text tokens are masked with `IGNORE_ID`)
- **Optimizer**: Adam, lr=1e-5 (for fine-tuning)
- **Scheduler**: Constant LR with 2500 warmup steps
- **Gradient clipping**: max_norm=5
- **Gradient accumulation**: 2 steps
- **Mixed precision**: AMP (fp16)
- **Checkpoint**: initialized from pretrained Fun-CosyVoice3-0.5B

### Flow Training
- Same optimizer/scheduler settings as LLM
- **Loss**: MSE between predicted and ground-truth velocity field (conditional flow matching loss)
- **Unified training**: 50% streaming (chunk-masked attention) + 50% non-streaming (full attention) per batch
- **Condition dropout**: 50% of samples have a random prefix of the target mel used as condition; 20% CFG dropout of all conditions

### HiFi-GAN Training
- **GAN training** with alternating generator/discriminator steps
- **Generator loss**: L1 mel reconstruction + feature matching + adversarial loss
- **Discriminator**: Multi-Period Discriminator (MPD) + Multi-Resolution Spectral Discriminator (MRSD)
- **Optimizer**: Adam, lr=0.0002
- `accum_grad=1` (no gradient accumulation for GAN)

In [ ]:
# File: examples/libritts/cosyvoice3/run.sh — Training command (Stage 5)

"""
# Train each component sequentially: llm, flow, hifigan
for model in llm flow hifigan; do
    torchrun --nnodes=1 --nproc_per_node=$num_gpus \
        --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
      cosyvoice/bin/train.py \
      --train_engine torch_ddp \
      --config conf/cosyvoice3.yaml \
      --train_data data/train.data.list \
      --cv_data data/dev.data.list \
      --qwen_pretrain_path $pretrained_model_dir/CosyVoice-BlankEN \
      --onnx_path $pretrained_model_dir \
      --model $model \                              # which component to train
      --checkpoint $pretrained_model_dir/$model.pt \ # initialize from pretrained
      --model_dir exp/cosyvoice3/$model/torch_ddp \
      --tensorboard_dir tensorboard/cosyvoice3/$model/torch_ddp \
      --use_amp                                      # mixed precision training
done
"""

# Training configuration summary (from cosyvoice3.yaml)
train_conf = {
    "optim": "adam",
    "lr": 1e-5,                  # fine-tuning LR
    "scheduler": "constantlr",
    "warmup_steps": 2500,
    "max_epoch": 200,
    "grad_clip": 5,
    "accum_grad": 2,             # effective batch = 2 * actual batch
    "log_interval": 100,
}

train_conf_gan = {
    "optim": "adam",     "lr": 0.0002,
    "optim_d": "adam",   "lr_d": 0.0002,   # discriminator
    "accum_grad": 1,     # no accumulation for GAN
}

## 7.4 Post-Training: DPO and GRPO

CosyVoice 3 supports **reinforcement learning from human feedback (RLHF)** style post-training:

### DPO (Direct Preference Optimization)
The LLM supports `forward_dpo()` which computes DPO loss by comparing chosen vs. rejected speech token sequences for the same text. This requires paired data with preferred and rejected speech samples.

### GRPO (Group Relative Policy Optimization)
The repository includes a complete GRPO training setup in `examples/grpo/cosyvoice2/`. This uses:
- A **reward server** (`token2wav_asr_server.py`) that converts speech tokens -> waveform -> ASR transcription and computes a text similarity reward
- The reward signal guides the LLM to produce more intelligible speech tokens

The RL-trained model (`Fun-CosyVoice3-0.5B-2512_RL`) achieves significantly better content consistency: **0.81% CER** (vs 1.21% for the base model) on Chinese and **1.68% WER** (vs 2.24%) on English.

---

# 8. Usage Examples

Here are the main ways to use CosyVoice 3 for inference:

In [ ]:
import sys
sys.path.append('third_party/Matcha-TTS')
from cosyvoice.cli.cosyvoice import AutoModel
import torchaudio

# Load model (auto-detects CosyVoice3 from cosyvoice3.yaml in model_dir)
cosyvoice = AutoModel(model_dir='pretrained_models/Fun-CosyVoice3-0.5B')

# --- Zero-shot voice cloning ---
# Clone voice from a reference audio, synthesize new text
for i, j in enumerate(cosyvoice.inference_zero_shot(
    '八百标兵奔北坡，北坡炮兵并排跑。',                    # text to synthesize
    'You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。',  # prompt text
    './asset/zero_shot_prompt.wav',                        # reference audio
    stream=False
)):
    torchaudio.save(f'zero_shot_{i}.wav', j['tts_speech'], cosyvoice.sample_rate)

# --- Instruct mode (control language/dialect/emotion/speed) ---
for i, j in enumerate(cosyvoice.inference_instruct2(
    '好少咯，一般系放嗰啲国庆啊，中秋嗰啲可能会咯。',     # text (Cantonese content)
    'You are a helpful assistant. 请用广东话表达。<|endofprompt|>',  # instruction
    './asset/zero_shot_prompt.wav',                        # reference voice
    stream=False
)):
    torchaudio.save(f'instruct_{i}.wav', j['tts_speech'], cosyvoice.sample_rate)

# --- Pronunciation inpainting ---
# Force specific pronunciation with phoneme tokens
for i, j in enumerate(cosyvoice.inference_zero_shot(
    '高管也通过电话、短信、微信等方式对报道[j][ǐ]予好评。',  # [j][ǐ] overrides pronunciation
    'You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。',
    './asset/zero_shot_prompt.wav', stream=False
)):
    torchaudio.save(f'hotfix_{i}.wav', j['tts_speech'], cosyvoice.sample_rate)

# --- Bi-streaming (text from LLM, speech generated progressively) ---
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'

for i, j in enumerate(cosyvoice.inference_zero_shot(
    text_generator(),  # streaming text input
    '希望你以后能够做的比我还好呦。',
    './asset/zero_shot_prompt.wav', stream=True  # streaming output
)):
    torchaudio.save(f'bistream_{i}.wav', j['tts_speech'], cosyvoice.sample_rate)

---

# 9. CosyVoice Evolution: v1 vs v2 vs v3

| Feature | CosyVoice v1 | CosyVoice v2 | CosyVoice v3 |
|---|---|---|---|
| **LLM Backbone** | Custom Transformer + Conformer text encoder | Qwen2ForCausalLM (0.5B) | Qwen2ForCausalLM (0.5B) |
| **LLM Class** | `TransformerLM` | `Qwen2LM` | `CosyVoice3LM` |
| **Speech Token Vocab** | 4096 (v1 tokenizer) | 4096 (v2 tokenizer) | **6561** (FSQ v3 tokenizer) |
| **Token Frame Rate** | 50 Hz | 25 Hz | 25 Hz |
| **Flow Architecture** | `MaskedDiffWithXvec` (Conformer encoder + UNet decoder) | `CausalMaskedDiffWithXvec` (Conformer encoder + UNet decoder) | **`CausalMaskedDiffWithDiT`** (PreLookahead + **DiT** decoder) |
| **Flow Encoder** | UpsampleConformerEncoder | UpsampleConformerEncoder | **PreLookaheadLayer** (lightweight Conv1d) |
| **Vocoder** | HiFTGenerator | CausalHiFTGenerator | CausalHiFTGenerator |
| **Sample Rate** | 22050 Hz | 24000 Hz | 24000 Hz |
| **Streaming** | Limited (mel overlap) | Full (causal + chunk) | Full (causal + chunk) |
| **Bistream** | No | Yes | Yes |
| **Instruct** | Separate model | Via prompt | **Via system prompt** (`<\|endofprompt\|>`) |
| **Pronunciation Control** | No | No | **Yes** (CMU + Pinyin tokens) |
| **Speaker Embedding** | In LLM sequence | In LLM sequence | **Not in LLM** (flow only) |
| **RL Post-training** | No | GRPO | DPO + GRPO |

---

# 10. File Structure Summary

```
CosyVoice/
├── cosyvoice/
│   ├── cli/
│   │   ├── cosyvoice.py        # Top-level API: CosyVoice, CosyVoice2, CosyVoice3, AutoModel
│   │   ├── model.py            # CosyVoiceModel, CosyVoice2Model, CosyVoice3Model (orchestration)
│   │   └── frontend.py         # CosyVoiceFrontEnd (text normalization, tokenization, embedding)
│   ├── llm/
│   │   └── llm.py              # TransformerLM, Qwen2LM, CosyVoice3LM, Qwen2Encoder
│   ├── flow/
│   │   ├── flow.py             # MaskedDiffWithXvec, CausalMaskedDiffWithXvec, CausalMaskedDiffWithDiT
│   │   ├── flow_matching.py    # ConditionalCFM, CausalConditionalCFM (ODE solver)
│   │   └── DiT/
│   │       ├── dit.py          # DiT backbone (TextEmbedding, InputEmbedding, DiT)
│   │       └── modules.py      # TimestepEmbedding, DiTBlock, ConvPositionEmbedding, etc.
│   ├── hifigan/
│   │   ├── generator.py        # HiFTGenerator, CausalHiFTGenerator, SineGen, ResBlock
│   │   ├── discriminator.py    # MultipleDiscriminator (MPD + MRSD)
│   │   └── f0_predictor.py     # CausalConvRNNF0Predictor
│   ├── tokenizer/
│   │   └── tokenizer.py        # CosyVoice2Tokenizer, CosyVoice3Tokenizer, get_qwen_tokenizer
│   ├── transformer/
│   │   ├── upsample_encoder.py # PreLookaheadLayer, UpsampleConformerEncoder
│   │   ├── encoder.py          # ConformerEncoder
│   │   └── attention.py        # Multi-head attention implementations
│   ├── dataset/
│   │   ├── processor.py        # Data processing functions (parquet_opener, filter, tokenize, etc.)
│   │   └── dataset.py          # Dataset class
│   ├── bin/
│   │   └── train.py            # Training entry point
│   └── utils/
│       ├── executor.py         # Training loop (Executor)
│       ├── train_utils.py      # DDP init, optimizer, scheduler, model saving
│       └── common.py           # Utilities (ras_sampling, fade_in_out, etc.)
├── examples/
│   └── libritts/cosyvoice3/
│       ├── conf/cosyvoice3.yaml  # Model + training configuration
│       └── run.sh                # End-to-end training script
├── example.py                    # Usage examples for all 3 versions
└── third_party/Matcha-TTS/       # Base flow matching implementation
```

---

# 11. References

1. **CosyVoice 3 paper**: Du et al., "CosyVoice 3: Towards In-the-wild Speech Generation via Scaling-up and Post-training", arXiv:2505.17589, 2025
2. **CosyVoice 2 paper**: Du et al., "CosyVoice 2: Scalable streaming speech synthesis with large language models", arXiv:2412.10117, 2024
3. **CosyVoice 1 paper**: Du et al., "CosyVoice: A scalable multilingual zero-shot text-to-speech synthesizer based on supervised semantic tokens", arXiv:2407.05407, 2024
4. **Model weights**: [HuggingFace: FunAudioLLM/Fun-CosyVoice3-0.5B-2512](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512)
5. **Evaluation**: [CV3-Eval](https://github.com/FunAudioLLM/CV3-Eval)